In [15]:
import pandas as pd
import numpy as np
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib
 
print("Imports done")

Imports done


In [7]:
RAW_PATH = r"C:\Users\mansi.apet\Downloads\AI_SYSTEM (2)\AI_System(2)\AI_System_New_1\data\raw\subscription_dataset_large.csv"
RECURRING_PATH = r"C:\Users\mansi.apet\Downloads\AI_SYSTEM (2)\AI_System(2)\AI_System_New_1\data\processed\subscription_with_patterns.csv"
 
df_raw = pd.read_csv(RAW_PATH, parse_dates=["Date"])
df_rec = pd.read_csv(RECURRING_PATH)[["TransactionID", "is_recurring"]]
 
df = df_raw.merge(df_rec, on="TransactionID", how="left")
 
df["is_recurring"] = df["is_recurring"].fillna(False)
df = df.rename(columns={"SubscriptionFlag": "is_subscription"})
 
df.head()

,CustomerID,TransactionID,Date,Description,Amount,Balance,Merchant,is_subscription,Status,is_recurring
0,1,15,2024-01-01,Gym Membership Subscription,300,9880,Gym Membership,1,SUCCESS,True
1,1,1,2024-01-05,Netflix Subscription,499,14567,Netflix,1,SUCCESS,True
2,1,2,2024-01-05,Netflix Subscription,499,14068,Netflix,1,SUCCESS,True
3,1,16,2024-01-08,Gym Membership Subscription,300,9580,Gym Membership,1,SUCCESS,True
4,1,41,2024-01-08,Electricity Bill Subscription,1200,1180,Electricity Bill,1,SUCCESS,False


In [8]:
customer_df = df.groupby("CustomerID").agg(
    total_spent=("Amount", "sum"),
    avg_amount=("Amount", "mean"),
    num_transactions=("Amount", "count"),
    num_subscriptions=("is_subscription", "sum"),
    num_recurring=("is_recurring", "sum"),
    num_failed=("Status", lambda x: (x == "FAILED").sum()),
    avg_balance=("Balance", "mean"),
    min_balance=("Balance", "min"),
).reset_index()
 
customer_df["failed_ratio"] = (
    customer_df["num_failed"] / customer_df["num_transactions"]
)
 
customer_df.head()

,CustomerID,total_spent,avg_amount,num_transactions,num_subscriptions,num_recurring,num_failed,avg_balance,min_balance,failed_ratio
0,1,48163,625.493506,77,47,40,3,-2601.870130,-33097,0.038961
1,2,47427,615.935065,77,47,14,6,-10984.285714,-40592,0.077922
2,3,49397,641.519481,77,47,14,3,-6571.610390,-37949,0.038961
3,4,49795,646.688312,77,47,14,11,-10366.597403,-42945,0.142857
4,5,50555,656.558442,77,47,40,3,-2541.883117,-34183,0.038961


In [9]:
# thresholds
TOTAL_SPENT_THRESHOLD = customer_df["total_spent"].quantile(0.9)
AVG_AMOUNT_THRESHOLD = customer_df["avg_amount"].quantile(0.9)
 
customer_df["risk_label"] = (
    (customer_df["num_failed"] > 2) |
    (customer_df["total_spent"] > TOTAL_SPENT_THRESHOLD) |
    (customer_df["avg_amount"] > AVG_AMOUNT_THRESHOLD)
).astype(int)
 
customer_df["risk_label"].value_counts()

risk_label
1    912
0    588
Name: count, dtype: int64

In [10]:
FEATURES = [
    "total_spent", "avg_amount", "num_transactions",
    "num_subscriptions", "num_recurring",
    "num_failed", "failed_ratio",
    "avg_balance", "min_balance"
]
 
X = customer_df[FEATURES]
y = customer_df["risk_label"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000,class_weight='balanced')
lr.fit(X_train_scaled, y_train)
 
# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=6,class_weight='balanced')
rf.fit(X_train, y_train)
 
print("Models trained")

Models trained


In [12]:
def evaluate(model, X, y):
    y_pred = model.predict(X)
    return {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred),
        "recall": recall_score(y, y_pred),
        "f1": f1_score(y, y_pred)
    }
 
lr_metrics = evaluate(lr, X_test_scaled, y_test)
rf_metrics = evaluate(rf, X_test, y_test)
 
print("Logistic Regression:", lr_metrics)
print("Random Forest:", rf_metrics)

Logistic Regression: {'accuracy': 0.89, 'precision': 0.935672514619883, 'recall': 0.8791208791208791, 'f1': 0.9065155807365439}
Random Forest: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


In [13]:
if rf_metrics["f1"] >= lr_metrics["f1"]:
    best_model = rf
    use_scaler = False
    print("Best model: Random Forest")
else:
    best_model = lr
    use_scaler = True
    print("Best model: Logistic Regression")

Best model: Random Forest


In [14]:
X_all = scaler.transform(X) if use_scaler else X
 
risk_scores = best_model.predict_proba(X_all)[:, 1]
 
customer_df["risk_score"] = risk_scores
customer_df["risk_class"] = (risk_scores > 0.7).astype(int)
 
customer_df[["CustomerID", "risk_score", "risk_class"]].head()
 

,CustomerID,risk_score,risk_class
0,1,0.991238,1
1,2,1.000000,1
2,3,0.999000,1
3,4,1.000000,1
4,5,0.991238,1


In [17]:
def calculate_risk(df):
    return customer_df[["CustomerID", "risk_score", "risk_class"]].copy()

risk_df = calculate_risk(df)
print(risk_df.head())

   CustomerID  risk_score  risk_class
0           1    0.991238           1
1           2    1.000000           1
2           3    0.999000           1
3           4    1.000000           1
4           5    0.991238           1


In [20]:
# Merge risk_score and risk_class back onto the full customer features
final_output = customer_df.copy()  # already has all feature columns

final_output.to_csv(
    r"C:\Users\mansi.apet\Downloads\AI_SYSTEM (2)\AI_System(2)\AI_System_New_1\data\output\final_dataset_with_risk.csv",
    index=False
)
print("Saved:", final_output.shape)
print(final_output.columns.tolist())

Saved: (1500, 13)
['CustomerID', 'total_spent', 'avg_amount', 'num_transactions', 'num_subscriptions', 'num_recurring', 'num_failed', 'avg_balance', 'min_balance', 'failed_ratio', 'risk_label', 'risk_score', 'risk_class']


In [21]:
joblib.dump(best_model, 
    r"C:\Users\mansi.apet\Downloads\AI_SYSTEM (2)\AI_System(2)\AI_System_New_1\models\risk_model.pkl"
)
print("Model saved")

Model saved


In [23]:
# def predict_new_customer(new_df):
#     # new_df = transactions of the new customer handed to you live

#     # Step 1 - build features same way as training
#     cust = new_df.groupby("CustomerID").agg(
#         total_spent       = ("Amount", "sum"),
#         avg_amount        = ("Amount", "mean"),
#         num_transactions  = ("Amount", "count"),
#         num_subscriptions = ("is_subscription", "sum"),
#         num_recurring     = ("is_recurring", "sum"),
#         num_failed        = ("Status", lambda x: (x == "FAILED").sum()),
#         avg_balance       = ("Balance", "mean"),
#         min_balance       = ("Balance", "min"),
#     ).reset_index()

#     cust["failed_ratio"] = cust["num_failed"] / cust["num_transactions"]

#     # Step 2 - load saved model
#     model = joblib.load(r"...\models\risk_model.pkl")

#     # Step 3 - predict
#     FEATURES = ["total_spent", "avg_amount", "num_transactions",
#                 "num_subscriptions", "num_recurring", "num_failed",
#                 "failed_ratio", "avg_balance", "min_balance"]

#     risk_score = model.predict_proba(cust[FEATURES])[:, 1]
#     risk_class = (risk_score > 0.7).astype(int)

#     cust["risk_score"] = risk_score.round(4)
#     cust["risk_class"] = risk_class

#     return cust[["CustomerID", "risk_score", "risk_class"]]


# # Test it with the evaluator's live data
# result = predict_new_customer(new_customer_df)
#  print(result)